In [ ]:
import random
import joblib
from nn_models   import *
from nn_data     import *
from nn_training import *
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

In [ ]:
DATA_PATH    = '/path'
BEST_HP_PATH = '/path'

### 1. Load data

In [ ]:
features = KEY_FEATURES + CONTROL_FEATURES
df_train = pd.read_csv(DATA_PATH + 'train.csv')
df_test  = pd.read_csv(DATA_PATH + 'test.csv')

Xy_train = df_train.copy()
Xy_test  = df_test.copy()

### 2. Standardize data

In [ ]:
# fit statistics on training set
feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

feature_scaler.fit(Xy_train[features])
target_scaler.fit(Xy_train['pm25_filled'].values.reshape(-1, 1))

# Apply transformation
Xy_train[features]      = feature_scaler.transform(Xy_train[features])
Xy_train['pm25_filled'] = target_scaler.transform(Xy_train['pm25_filled'].values.reshape(-1, 1))
Xy_test[features]       = feature_scaler.transform(Xy_test[features])
Xy_test['pm25']         = transform_target_preserve_nan(Xy_test['pm25'], target_scaler) 

# Group data based on time sequence
train_groups = prepare_3d_input(df=Xy_train, ts_cols=features, target_col='pm25_filled', verbose=False)
test_groups  = prepare_3d_input(df=Xy_test,  ts_cols=features, verbose=False)

train_tensors = groups_to_tensors(train_groups, mode='cnn1d', reverse_sort=True)

# Quick check
for T, (X_t, y_t) in train_tensors.items():
    print(f"Train group T={T:4d} | X_t: {tuple(X_t.shape)} |y: {tuple(y_t.shape)}")

### 3. Train the model

In [ ]:
loaded_study = joblib.load(BEST_HP_PATH)
best         = loaded_study.best_params
best_trial   = loaded_study.best_trial
best_epoch   = best_trial.user_attrs["best_epoch"]

In [ ]:
seed_everything()
n_features = len(features)

model      = CNN1D(in_features  = n_features, 
                   h1           = best_trial.user_attrs['h1'], 
                   h2           = best_trial.user_attrs['h2'], 
                   hout         = best_trial.user_attrs['hout'], 
                   drop_rate    = best['drop_rate'])

optimizer = torch.optim.Adam(model.parameters(), lr=best['lr'])
criterion = nn.MSELoss()

epochs = best_epoch
train_losses = train_CNN1D(model, train_tensors, criterion, optimizer, epochs) # X: [B, T, F]

### 4. Collect predictions

In [ ]:
df_test_pred  = collect_CNN1D_preds(test_groups,  
                                    df_test,  
                                    model, 
                                    target_scaler)